In [ ]:
#pip install ollama

In [1]:
import ollama
import pandas as pd
from datetime import datetime
import time

Ask Gwen3 if it can print all 78 tarot cards?

ToDo

- write to a csv every 10 rows
- run it in VSCode?

- skip every 100 seconds? 



csv_file = "benchmark_results.csv"
buffer = []
flush_size = 10  # write every 10 runs (adjust as needed)

# Create CSV with header if it doesn't exist
if not os.path.exists(csv_file):
    pd.DataFrame(columns=[
        "run",
        "timestamp",
        "response",
        "duration_sec",
        "response_tokens",
        "model"
    ]).to_csv(csv_file, index=False)


    # Flush buffer to CSV every N runs
    if len(buffer) >= flush_size:
        pd.DataFrame(buffer).to_csv(csv_file, mode="a", header=False, index=False)
        buffer = []

# Flush any remaining rows
if buffer:
    pd.DataFrame(buffer).to_csv(csv_file, mode="a", header=False, index=False)


In [2]:
model = 'qwen3:4b'

### Other Models
#model = 'qwen3:1.7b'
#model = 'llama3.2:3b'

In [3]:
tarot_prompt = """
Please list ALL 78 tarot cards from the Rider-Waite deck. Respond concisely. 
Do not show reasoning steps. Only provide the names of the 78 card in text. 
"""

tarot_message = [{"role": "user", "content": tarot_prompt}]


start_time = time.perf_counter()

tarot_response = ollama.chat(
        model=model,
        messages=tarot_message)

end_time = time.perf_counter()
duration = end_time - start_time
print(f"Time taken: {duration:.2f} seconds")

Time taken: 170.17 seconds


In [4]:
tarot_str = tarot_response['message']['content']
tarot_list = [card.strip() for card in tarot_str.split(",")]

print(len(tarot_list))
print()

for card in tarot_list:
    print(card)

74

The Fool
The Magician
The High Priestess
The Empress
The Emperor
The Hierophant
The Lovers
The Chariot
Strength
The Hermit
The Wheel of Fortune
Justice
The Hanged Man
Death
Temperance
The Devil
The Tower
The Star
The Moon
The Sun
The Judgment
The World
Ace of Wands
2 of Wands
3 of Wands
4 of Wands
5 of Wands
6 of Wands
7 of Wands
8 of Wands
9 of Wands
10 of Wands
Jack of Wands
Queen of Wands
King of Wands
Ace of Cups
2 of Cups
3 of Cups
4 of Cups
5 of Cups
6 of Cups
7 of Cups
8 of Cups
9 of Cups
10 of Cups
Jack of Cups
Queen of Cups
King of Cups
Ace of Swords
2 of Swords
3 of Swords
4 of Swords
5 of Swords
6 of Swords
7 of Swords
8 of Swords
9 of Swords
10 of Swords
Jack of Swords
Queen of Swords
King of Swords
Ace of Pentacles
2 of Pentacles
3 of Pentacles
4 of Pentacles
5 of Pentacles
6 of Pentacles
7 of Pentacles
8 of Pentacles
9 of Pentacles
10 of Pentacles
Jack of Pentacles
Queen of Pentacles
King of Pentacles


In [5]:
#format

In [6]:
text = """Randomly select a tarot card from the full Rider-Waite deck (Major & Minor Arcana).
Respond concisely. Do not show reasoning steps. Only provide the name of the card in text. 
"""

messages = [{"role": "user", "content": text}]

In [7]:
#
loops = 10
token_limit = 1000

In [8]:
#No Limit
total_start = time.perf_counter()

results = []

for i in range(loops):
    start_time = time.perf_counter()

    response = ollama.chat(
        model=model,
        messages=messages,
        options={"num_predict": token_limit} #,
                #"temperature": 0}
    )
    
    end_time = time.perf_counter()
    duration = end_time - start_time

    # Token counts returned by Ollama
    prompt_tokens = response.get("prompt_eval_count", 0)
    completion_tokens = response.get("eval_count", 0)
    total_tokens = prompt_tokens + completion_tokens
    
    #if (i + 1) % 10 == 0:
    print(
        f"Run {i+1} {response['message']['content']} took {duration:.2f} seconds "
        f"({total_tokens} tokens: {prompt_tokens} prompt + {completion_tokens} response)"
    )

    results.append({
        "run": i + 1,
        "timestamp": datetime.now().isoformat(),
        "response": response["message"]["content"],
        "duration_sec": duration,
        #"prompt_tokens": prompt_tokens,
        "response_tokens": completion_tokens,
        #"total_tokens": total_tokens,
        "model":model
    })

# End timing the entire benchmark
total_duration = time.perf_counter() - total_start

df = pd.DataFrame(results)

print(df)
print(f"\nTotal benchmark time: {total_duration:.2f} seconds")

Run 1 The Tower took 157.07 seconds (1041 tokens: 54 prompt + 987 response)
Run 2  took 164.20 seconds (1054 tokens: 54 prompt + 1000 response)
Run 3  took 160.48 seconds (1054 tokens: 54 prompt + 1000 response)
Run 4 The Lovers took 48.11 seconds (399 tokens: 54 prompt + 345 response)
Run 5 The Sun took 37.77 seconds (320 tokens: 54 prompt + 266 response)
Run 6  took 156.90 seconds (1054 tokens: 54 prompt + 1000 response)
Run 7  took 169.86 seconds (1054 tokens: 54 prompt + 1000 response)
Run 8 The Empress took 43.28 seconds (360 tokens: 54 prompt + 306 response)
Run 9 The Moon took 128.20 seconds (823 tokens: 54 prompt + 769 response)
Run 10  took 152.00 seconds (1054 tokens: 54 prompt + 1000 response)
   run                   timestamp     response  duration_sec  \
0    1  2026-07-04T11:09:02.978119    The Tower    157.067220   
1    2  2026-07-04T11:11:47.187554                 164.204794   
2    3  2026-07-04T11:14:27.675187                 160.483110   
3    4  2026-07-04T11:15:1

In [9]:
df.head(10)

,run,timestamp,response,duration_sec,response_tokens,model
0,1,2026-07-04T11:09:02.978119,The Tower,157.067220,987,qwen3:4b
1,2,2026-07-04T11:11:47.187554,,164.204794,1000,qwen3:4b
2,3,2026-07-04T11:14:27.675187,,160.483110,1000,qwen3:4b
3,4,2026-07-04T11:15:15.787190,The Lovers,48.110571,345,qwen3:4b
4,5,2026-07-04T11:15:53.560766,The Sun,37.772377,266,qwen3:4b
5,6,2026-07-04T11:18:30.466696,,156.901467,1000,qwen3:4b
6,7,2026-07-04T11:21:20.333808,,169.862329,1000,qwen3:4b
7,8,2026-07-04T11:22:03.619537,The Empress,43.284422,306,qwen3:4b
8,9,2026-07-04T11:24:11.822683,The Moon,128.199508,769,qwen3:4b
9,10,2026-07-04T11:26:43.823425,,151.996454,1000,qwen3:4b


In [10]:
df['response'].value_counts()

               5
The Tower      1
The Lovers     1
The Sun        1
The Empress    1
The Moon       1
Name: response, dtype: int64